### 14개 LOT의 공통 움직임을 확인하고 1분 v5 만들기

- `TC-Z3`, `TC-Z4`, `CP-Z4`, `TC-Z5`, `TC-Z6`와 대응 출력을 함께 비교함.
  - 같은 시각에 똑같이 움직이는지보다, 올라가고 내려오는 순서와 회복을 확인함.
  - 확인한 센서와 시간만 후보에서 해제함.
- `v3`의 값과 라벨은 그대로 유지함.
  - `v5`는 실제 관측한 구간 안에서 1분씩 묶어 저장함.
  - 빈 날이나 빠진 초는 만들지 않고 보간하지 않음.
- 통계적 라벨과 도메인 검토 라벨을 따로 남김.
  - 아직 검토하지 않은 구간을 정상 `0`으로 채우지 않음.
  - 이번 판단은 사후 검토이며 실제 고장이나 품질의 정답은 아님.


### 1. 읽을 파일과 저장할 위치

- `cal01_base_v3.csv`와 함께 만든 `metadata/v3/dataset.json`이 필요함.
- 조업이벤트와 정비이력은 `data/` 또는 `data/raw/`에 둠.
- `SAVE_FILES=True`이면 `cal01_base_v5.csv`, 변경표, 그림을 저장함.
- 센서값 보간, 빈 초 생성, 학습용 표본 추출은 하지 않음.
- 최종 `cal01_base_v5.csv`는 1분 집계임.
  - 원자료 라벨 변경표와 원자료-분 연결표는 `review/v5`에 남김.


In [ ]:
INPUT_FILE = "cal01_base_v3.csv"
OUTPUT_FILE = "cal01_base_v5.csv"
SAVE_FILES = True

import os
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager
from IPython.display import display

ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)

DATA_DIR = os.path.join(ROOT, "data")
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")

PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
INPUT_CSV = os.path.join(PROCESSED_DIR, INPUT_FILE)
OUTPUT_CSV = os.path.join(PROCESSED_DIR, OUTPUT_FILE)
META_DIR = os.path.join(PROCESSED_DIR, "metadata", "v5")
REVIEW_DIR = os.path.join(PROCESSED_DIR, "review", "v5")
FIG_DIR = os.path.join(ROOT, "figures", "namelesspark", "16_end_pattern_v5")
SOURCE_META = os.path.join(PROCESSED_DIR, "metadata", "v3", "dataset.json")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
ENC = "utf-8-sig"

# 직접 비교한 v3에만 아래 시간표를 적용함. 다른 CSV에는 자동으로 해제하지 않음.
REVIEWED_SHA256 = "247c6f328149427e93a588925a1e7b27a40cf3f9b3922997477d104d77094aa8"
with open(INPUT_CSV, "rb") as file:
    source_sha256 = hashlib.file_digest(file, "sha256").hexdigest()
with open(SOURCE_META, encoding="utf-8") as file:
    source_metadata = json.load(file)

review_applied = source_sha256 == REVIEWED_SHA256 and source_metadata.get("master_sha256") == source_sha256
# 경로를 바꾸더라도 v3를 덮어쓰지 않음.
write_allowed = SAVE_FILES and review_applied and os.path.realpath(INPUT_CSV) != os.path.realpath(OUTPUT_CSV)
if SAVE_FILES and not write_allowed:
    print("저장 보류: 검토한 v3와 생성 정보가 맞는지, 출력 경로가 입력과 다른지 확인하세요.")

# 문자열로 읽어 원본 값의 자릿수와 빈칸까지 유지함. 계산할 값만 따로 숫자로 바꿈.
base = pd.read_csv(INPUT_CSV, encoding=ENC, dtype=str, keep_default_na=False)
output = base.copy()
times = pd.to_datetime(base["MEAS_DT"])
not_excluded = base["EXCLUDED"].eq("False")
groups = source_metadata["groups"]
sensors = source_metadata["raw_sensor_columns"]
values = base[sensors].apply(pd.to_numeric, errors="coerce")
lots = sorted(base["LOT_NO"].unique())
lot_end = times.groupby(base["LOT_NO"]).max()
lot_start = times.groupby(base["LOT_NO"]).min()

# v3와 같은 LOT별 기준. 기존 제외는 평균과 표준편차 계산에서도 뺌.
reference = values.loc[not_excluded]
lot_mean = reference.groupby(base.loc[not_excluded, "LOT_NO"]).mean()
lot_std = reference.groupby(base.loc[not_excluded, "LOT_NO"]).std(ddof=0)

events = pd.read_csv(EVENT_CSV, encoding=ENC)
events["EVT_DT"] = pd.to_datetime(events["EVT_DT"])
events = events.loc[events["PLANT_CD"].eq("CR1") & events["EVT_TYPE"].eq("강종변경")].copy()
maintenance = pd.read_csv(MAINT_CSV, encoding=ENC)
maintenance = maintenance.loc[maintenance["EQP_TAG"].eq("P1-CR1-CAL01")].copy()
maintenance["STRT_DT"] = pd.to_datetime(maintenance["STRT_DT"])
maintenance["END_DT"] = pd.to_datetime(maintenance["END_DT"])

font_names = []
for font in font_manager.fontManager.ttflist:
    font_names.append(font.name)
for font_name in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if font_name in font_names:
        plt.rcParams["font.family"] = font_name
        break
plt.rcParams["axes.unicode_minus"] = False

if write_allowed:
    for folder in [META_DIR, REVIEW_DIR, FIG_DIR]:
        os.makedirs(folder, exist_ok=True)
print("입력 행:", len(base), "| LOT:", len(lots), "| 검토표 적용:", review_applied)

### 2. 14개 LOT의 마지막 2시간을 같은 위치에서 보기

- 가로축 `0`은 LOT의 마지막 기록임. `-20`은 마지막 기록 20분 전임.
- 표의 날짜와 시각은 실제 기록 시각임. 고점은 마지막 35분의 1분 평균 중 가장 높은 값임.
- 위 비교 그림은 1분 평균임. 아래에서 원자료 라벨을 검토한 뒤 최종 v5도 1분으로 집계함.
- `058`, `114`는 온도와 출력이 낮아진 채 끝나므로 이번 해제에서 뺌.
- 나머지 12개 LOT는 건조 온도가 올라갔다 내려가고 출력이 낮아졌다 다시 올라가는 모습이 반복됨.
  - 일부 LOT는 온도가 완전히 회복하기 전에 기록이 끝남. 회복 완료나 정상 운전이 확인됐다는 뜻은 아님.
- `Z3`, `Z6`는 4-2번의 069 추가 범위만 수정함. 마지막 2시간 전체를 0으로 바꾸지 않음.

In [ ]:
lot_review_rows = []
for lot in lots:
    index = base.index[base["LOT_NO"].eq(lot)]
    last_index = index[times.loc[index].ge(lot_end[lot] - pd.Timedelta(minutes=35))]
    minute_values = values.loc[last_index].copy()
    minute_values.index = pd.DatetimeIndex(times.loc[last_index])
    minute_values = minute_values.resample("1min").mean()

    row = {
        "LOT": lot,
        "강종": base.loc[index[0], "GRADE"],
        "마지막 기록": lot_end[lot],
        "이번 처리": "건조 구역의 확인 범위만 해제",
    }
    for sensor in ["TC-Z1", "TC-Z2"]:
        peak = minute_values[sensor].idxmax()
        row[sensor + " 고점 시각"] = peak
        row[sensor + " 고점 ℃"] = round(float(minute_values.loc[peak, sensor]), 2)
        row[sensor + " 고점 후 최저 ℃"] = round(float(minute_values.loc[peak:, sensor].min()), 2)
    if lot in ["240058", "240114"]:
        row["이번 처리"] = "온도와 출력 하강이 이어짐, 기존 판단 유지"
    lot_review_rows.append(row)

lot_review = pd.DataFrame(lot_review_rows)
display(lot_review)

# 센서별로 14개 LOT를 모두 보여줌. 15초보다 큰 공백은 선을 잇지 않음.
overview_sensors = ["TC-Z1", "TC-Z2", "OP-Z1", "OP-Z2", "TC-Z3", "TC-Z6"]
for sensor in overview_sensors:
    fig, axes = plt.subplots(7, 2, figsize=(14, 18), sharex=True)
    for axis, lot in zip(axes.flat, lots):
        index = base.index[base["LOT_NO"].eq(lot)]
        index = index[times.loc[index].ge(lot_end[lot] - pd.Timedelta(minutes=120))]
        clock = times.loc[index]
        pieces = clock.diff().dt.total_seconds().gt(15).cumsum()
        color = "#b5643e" if lot in ["240058", "240114"] else "#3f718a"
        for piece in pieces.unique():
            part = index[pieces.eq(piece)]
            trace = pd.Series(values.loc[part, sensor].to_numpy(), index=pd.DatetimeIndex(times.loc[part]))
            average = trace.resample("1min").mean()
            relative_time = (average.index - lot_end[lot]).total_seconds() / 60
            axis.plot(relative_time, average, color=color, linewidth=1)
        axis.set_title(lot + " | " + base.loc[index[0], "GRADE"], fontsize=10)
        axis.set_xlim(-120, 0)
        axis.grid(alpha=0.2)
        axis.axvline(-35, color="#aaaaaa", linestyle=":", linewidth=0.7)
    for axis in axes[-1]:
        axis.set_xlabel("마지막 기록까지 남은 시간 (분, 0이 마지막 기록)")
    unit = "℃" if sensor.startswith("TC-") else "%"
    fig.suptitle(sensor + " (" + unit + ") | 마지막 2시간, 1분 평균", fontsize=15)
    fig.tight_layout(rect=[0, 0.02, 1, 0.97])
    fig.text(0.05, 0.01, "갈색 058·114는 이번 해제에서 제외. 점선 오른쪽은 건조 구역을 자세히 비교한 마지막 35분.", fontsize=10)
    if write_allowed:
        fig.savefig(os.path.join(FIG_DIR, "01_all_lots_" + sensor + ".png"), dpi=120)
    plt.show()
    plt.close(fig)

### 3. 후보를 해제할 센서와 시간을 적기

- 위 14개 LOT 그림과 마지막 35분의 원자료를 비교해 아래 범위를 정했음.
- 온도는 마지막 봉우리와 그 뒤 내려간 부분, 출력은 온도 상승에 함께 낮아진 부분만 포함함.
- `START`는 포함하고 `END`는 포함하지 않음.
- `high`는 평균보다 높은 후보, `low`는 평균보다 낮은 후보임.
  - 같은 시간에 반대 방향으로 튄 값은 해제하지 않음.
- 출력이 다시 높아질 때의 짧은 튐은 이번 해제에 넣지 않음.
  - `121 OP-Z1`의 회복 뒤 떨어져 있는 낮은 후보 3행도 유지함.
- 이 범위는 현재 v3를 보고 정한 것임. 다른 파일에 마지막 35분 규칙으로 자동 적용하지 않음.
- 069의 7/7 23:30 전후도 별도로 비교함.
  - Z3 봉우리 뒤 Z6 최저까지 약 11분으로, 이미 해제한 뒤쪽과 비슷했음.
  - Z4, CP, Z5는 움직임이 전부 같지 않아 함께 해제하지 않음.
- 다른 LOT의 앞부분 봉우리는 회복 모습과 센서 관계가 달라 이번에 일괄 해제하지 않음.


- `J`로 시작하는 범위는 이번에 Z3~Z6와 CP를 함께 보고 추가한 범위임.
  - 같은 구역의 출력도 함께 확인함.
  - CP가 크게 내려간 `024`, `117`과 계속 내려가는 `058`은 그 부분을 해제하지 않음.
  - `037`, `069`의 별도 CP 급등도 유지함.


In [ ]:
review_ranges = pd.DataFrame([
    ['240001', 'TC-Z1', '2024-01-02 14:44:33', '2024-01-02 14:49:17', 'high'],
    ['240001', 'TC-Z1', '2024-01-02 14:53:45', '2024-01-02 14:59:16', 'low'],
    ['240001', 'TC-Z2', '2024-01-02 14:45:11', '2024-01-02 14:52:15', 'high'],
    ['240001', 'TC-Z2', '2024-01-02 14:56:53', '2024-01-02 15:03:40', 'low'],
    ['240001', 'OP-Z1', '2024-01-02 14:43:14', '2024-01-02 14:51:08', 'low'],
    ['240001', 'OP-Z2', '2024-01-02 14:45:03', '2024-01-02 14:54:42', 'low'],
    ['240024', 'TC-Z1', '2024-03-08 06:09:15', '2024-03-08 06:16:07', 'high'],
    ['240024', 'TC-Z1', '2024-03-08 06:22:13', '2024-03-08 06:29:30', 'low'],
    ['240024', 'TC-Z2', '2024-03-08 06:10:28', '2024-03-08 06:19:11', 'high'],
    ['240024', 'TC-Z2', '2024-03-08 06:24:20', '2024-03-08 06:29:49', 'low'],
    ['240024', 'OP-Z1', '2024-03-08 06:08:28', '2024-03-08 06:19:51', 'low'],
    ['240024', 'OP-Z2', '2024-03-08 06:10:46', '2024-03-08 06:23:16', 'low'],
    ['240037', 'TC-Z1', '2024-04-14 09:09:14', '2024-04-14 09:15:34', 'high'],
    ['240037', 'TC-Z1', '2024-04-14 09:18:56', '2024-04-14 09:24:51', 'low'],
    ['240037', 'TC-Z2', '2024-04-14 09:10:33', '2024-04-14 09:18:25', 'high'],
    ['240037', 'TC-Z2', '2024-04-14 09:21:53', '2024-04-14 09:27:20', 'low'],
    ['240037', 'OP-Z1', '2024-04-14 09:08:50', '2024-04-14 09:17:28', 'low'],
    ['240037', 'OP-Z2', '2024-04-14 09:11:07', '2024-04-14 09:19:57', 'low'],
    ['240038', 'TC-Z1', '2024-04-16 13:26:01', '2024-04-16 13:27:36', 'high'],
    ['240038', 'TC-Z1', '2024-04-16 13:33:52', '2024-04-16 13:37:08', 'low'],
    ['240038', 'TC-Z2', '2024-04-16 13:36:58', '2024-04-16 13:39:56', 'low'],
    ['240038', 'OP-Z1', '2024-04-16 13:24:04', '2024-04-16 13:29:31', 'low'],
    ['240038', 'OP-Z2', '2024-04-16 13:27:48', '2024-04-16 13:27:49', 'low'],
    ['240044', 'OP-Z1', '2024-05-03 19:46:25', '2024-05-03 19:50:17', 'low'],
    ['240044', 'OP-Z2', '2024-05-03 19:48:41', '2024-05-03 19:52:12', 'low'],
    ['240061', 'TC-Z1', '2024-06-15 04:42:29', '2024-06-15 04:44:59', 'low'],
    ['240061', 'TC-Z2', '2024-06-15 04:35:53', '2024-06-15 04:37:05', 'high'],
    ['240061', 'TC-Z2', '2024-06-15 04:45:33', '2024-06-15 04:49:20', 'low'],
    ['240061', 'OP-Z1', '2024-06-15 04:31:12', '2024-06-15 04:37:48', 'low'],
    ['240061', 'OP-Z2', '2024-06-15 04:33:45', '2024-06-15 04:40:52', 'low'],
    ['240069', 'TC-Z1', '2024-07-08 06:09:42', '2024-07-08 06:15:28', 'high'],
    ['240069', 'TC-Z1', '2024-07-08 06:19:18', '2024-07-08 06:25:12', 'low'],
    ['240069', 'TC-Z2', '2024-07-08 06:11:10', '2024-07-08 06:18:00', 'high'],
    ['240069', 'TC-Z2', '2024-07-08 06:22:06', '2024-07-08 06:29:32', 'low'],
    ['240069', 'OP-Z1', '2024-07-08 06:08:17', '2024-07-08 06:17:44', 'low'],
    ['240069', 'OP-Z2', '2024-07-08 06:10:25', '2024-07-08 06:20:50', 'low'],
    ['240091', 'TC-Z1', '2024-09-09 08:24:57', '2024-09-09 08:32:09', 'high'],
    ['240091', 'TC-Z1', '2024-09-09 08:38:57', '2024-09-09 08:44:13', 'low'],
    ['240091', 'TC-Z2', '2024-09-09 08:26:11', '2024-09-09 08:36:18', 'high'],
    ['240091', 'TC-Z2', '2024-09-09 08:40:40', '2024-09-09 08:44:13', 'low'],
    ['240091', 'OP-Z1', '2024-09-09 08:24:12', '2024-09-09 08:37:29', 'low'],
    ['240091', 'OP-Z2', '2024-09-09 08:26:47', '2024-09-09 08:39:42', 'low'],
    ['240108', 'TC-Z1', '2024-10-18 11:31:28', '2024-10-18 11:36:20', 'high'],
    ['240108', 'TC-Z1', '2024-10-18 11:40:40', '2024-10-18 11:45:38', 'low'],
    ['240108', 'TC-Z2', '2024-10-18 11:32:58', '2024-10-18 11:38:23', 'high'],
    ['240108', 'TC-Z2', '2024-10-18 11:43:34', '2024-10-18 11:49:34', 'low'],
    ['240108', 'OP-Z1', '2024-10-18 11:30:05', '2024-10-18 11:37:46', 'low'],
    ['240108', 'OP-Z2', '2024-10-18 11:32:28', '2024-10-18 11:40:43', 'low'],
    ['240117', 'TC-Z1', '2024-11-13 14:14:40', '2024-11-13 14:20:14', 'high'],
    ['240117', 'TC-Z1', '2024-11-13 14:29:36', '2024-11-13 14:32:20', 'low'],
    ['240117', 'TC-Z2', '2024-11-13 14:15:38', '2024-11-13 14:23:42', 'high'],
    ['240117', 'TC-Z2', '2024-11-13 14:30:22', '2024-11-13 14:32:20', 'low'],
    ['240117', 'OP-Z1', '2024-11-13 14:14:26', '2024-11-13 14:23:11', 'low'],
    ['240117', 'OP-Z2', '2024-11-13 14:15:47', '2024-11-13 14:28:08', 'low'],
    ['240121', 'TC-Z1', '2024-11-24 17:57:10', '2024-11-24 18:03:40', 'high'],
    ['240121', 'TC-Z1', '2024-11-24 18:07:59', '2024-11-24 18:14:42', 'low'],
    ['240121', 'TC-Z2', '2024-11-24 17:58:24', '2024-11-24 18:06:40', 'high'],
    ['240121', 'TC-Z2', '2024-11-24 18:11:14', '2024-11-24 18:20:06', 'low'],
    ['240121', 'OP-Z1', '2024-11-24 17:55:49', '2024-11-24 18:06:56', 'low'],
    ['240121', 'OP-Z2', '2024-11-24 17:57:38', '2024-11-24 18:10:16', 'low'],
    ['240133', 'TC-Z1', '2024-12-28 16:27:08', '2024-12-28 16:29:08', 'high'],
    ['240133', 'TC-Z1', '2024-12-28 16:35:24', '2024-12-28 16:38:18', 'low'],
    ['240133', 'TC-Z2', '2024-12-28 16:39:24', '2024-12-28 16:42:21', 'low'],
    ['240133', 'OP-Z1', '2024-12-28 16:26:02', '2024-12-28 16:31:16', 'low'],
    ['240133', 'OP-Z2', '2024-12-28 16:26:42', '2024-12-28 16:33:01', 'low'],
], columns=["LOT_NO", "SENSOR", "START", "END", "DIRECTION"])
review_ranges["START"] = pd.to_datetime(review_ranges["START"])
review_ranges["END"] = pd.to_datetime(review_ranges["END"])
review_ranges.insert(0, "REVIEW_ID", ["E" + str(i).zfill(3) for i in range(1, len(review_ranges) + 1)])
review_ranges["REASON"] = "여러 LOT에서 반복된 건조 온도 상승 후 하강"
review_ranges.loc[review_ranges["SENSOR"].str.startswith("OP-"), "REASON"] = "건조 온도 상승에 함께 낮아졌다 회복하는 출력"
print("확인한 센서 시간 범위:", len(review_ranges), "| LOT:", review_ranges["LOT_NO"].nunique())
# 069의 앞부분과 이미 해제한 뒤쪽 움직임을 비교해서 추가한 범위임.
# 아래 7개는 이전 검토 범위임. 이번 Z4, CP, Z5 검토는 joint_ranges에 따로 남김.
early_ranges = pd.DataFrame([
    ["240069", "TC-Z3", "2024-07-07 23:30:39", "2024-07-07 23:35:33", "high"],
    ["240069", "TC-Z3", "2024-07-07 23:39:00", "2024-07-07 23:42:26", "low"],
    ["240069", "OP-Z3", "2024-07-07 23:29:34", "2024-07-07 23:35:04", "low"],
    ["240069", "TC-Z6", "2024-07-07 23:40:31", "2024-07-07 23:44:52", "low"],
    ["240069", "TC-Z6", "2024-07-07 23:49:39", "2024-07-07 23:53:28", "high"],
    ["240069", "OP-Z6", "2024-07-07 23:40:54", "2024-07-07 23:44:50", "high"],
    ["240069", "OP-Z6", "2024-07-07 23:48:59", "2024-07-07 23:52:44", "low"],
], columns=["LOT_NO", "SENSOR", "START", "END", "DIRECTION"])
early_ranges["START"] = pd.to_datetime(early_ranges["START"])
early_ranges["END"] = pd.to_datetime(early_ranges["END"])
early_ranges["REVIEW_ID"] = ["A" + str(i).zfill(3) for i in range(1, len(early_ranges) + 1)]
early_ranges["REASON"] = "069 뒤쪽과 같은 Z3 상승, 약 11분 뒤 Z6 하강과 반등을 앞부분에서도 확인"
review_ranges = pd.concat([review_ranges, early_ranges], ignore_index=True)
print("추가 검토 포함 센서 시간 범위:", len(review_ranges))


# 실제 원자료에서 확인한 센서, 방향, 시간만 적음. END 시각은 포함하지 않음.
joint_ranges = pd.DataFrame([
    ['J001', 'J001', '240001', 'TC-Z4', '2024-01-02 13:30:14', '2024-01-02 13:34:58', 'high', 'Z3 봉우리와 가깝게 올라간 Z4 온도'],
    ['J002', 'J001', '240001', 'OP-Z4', '2024-01-02 13:30:07', '2024-01-02 13:36:57', 'low', 'Z4 온도 상승 때 낮아진 대응 출력'],
    ['J003', 'J001', '240001', 'TC-Z5', '2024-01-02 13:40:30', '2024-01-02 13:40:31', 'low', 'Z5 상승 뒤 낮아진 부분'],
    ['J004', 'J001', '240001', 'CP-Z4', '2024-01-02 13:30:04', '2024-01-02 13:30:58', 'high', 'Z3 봉우리와 함께 나타나고 돌아온 짧은 CP 상승'],
    ['J005', 'J001', '240001', 'CP-Z4', '2024-01-02 13:32:14', '2024-01-02 13:32:37', 'low', 'Z3 하강 때 반복되고 회복된 CP 하강'],
    ['J006', 'J002', '240024', 'TC-Z4', '2024-03-08 04:49:50', '2024-03-08 04:50:32', 'low', 'Z3 상승 전 함께 낮아진 Z4 온도'],
    ['J007', 'J002', '240024', 'OP-Z4', '2024-03-08 05:00:52', '2024-03-08 05:06:10', 'high', 'Z4 온도 하강 때 올라간 대응 출력'],
    ['J008', 'J002', '240024', 'TC-Z5', '2024-03-08 04:48:06', '2024-03-08 04:50:22', 'low', 'Z3 상승 전 낮아졌다 회복한 Z5 온도'],
    ['J009', 'J002', '240024', 'TC-Z5', '2024-03-08 04:59:13', '2024-03-08 05:01:31', 'high', 'Z3 봉우리 뒤 올라온 Z5 온도'],
    ['J010', 'J002', '240024', 'OP-Z5', '2024-03-08 04:47:54', '2024-03-08 04:50:54', 'high', 'Z5 온도가 낮아질 때 올라간 출력'],
    ['J011', 'J002', '240024', 'OP-Z5', '2024-03-08 04:59:01', '2024-03-08 05:00:58', 'low', 'Z5 온도 상승 때 낮아진 출력'],
    ['J012', 'J002', '240024', 'CP-Z4', '2024-03-08 04:44:54', '2024-03-08 04:47:22', 'low', 'Z3 상승 전 여러 LOT에서 반복된 CP 하강'],
    ['J013', 'J002', '240024', 'CP-Z4', '2024-03-08 04:55:12', '2024-03-08 04:55:32', 'high', 'Z3 봉우리와 함께 나타나고 돌아온 짧은 CP 상승'],
    ['J014', 'J003', '240037', 'TC-Z4', '2024-04-14 07:56:03', '2024-04-14 07:58:05', 'high', 'Z3 봉우리와 가깝게 올라간 Z4 온도'],
    ['J015', 'J003', '240037', 'TC-Z4', '2024-04-14 08:03:55', '2024-04-14 08:05:31', 'low', 'Z3 하강 뒤 함께 낮아진 Z4 온도'],
    ['J016', 'J003', '240037', 'OP-Z4', '2024-04-14 07:54:45', '2024-04-14 07:57:49', 'low', 'Z4 온도 상승 때 낮아진 대응 출력'],
    ['J017', 'J003', '240037', 'TC-Z5', '2024-04-14 07:57:17', '2024-04-14 08:01:02', 'high', 'Z3 봉우리 뒤 올라온 Z5 온도'],
    ['J018', 'J003', '240037', 'OP-Z5', '2024-04-14 07:57:17', '2024-04-14 08:00:37', 'low', 'Z5 온도 상승 때 낮아진 출력'],
    ['J019', 'J004', '240038', 'TC-Z4', '2024-04-16 12:09:41', '2024-04-16 12:10:42', 'high', 'Z3 봉우리와 가깝게 올라간 Z4 온도'],
    ['J020', 'J004', '240038', 'OP-Z4', '2024-04-16 12:08:29', '2024-04-16 12:11:08', 'low', 'Z4 온도 상승 때 낮아진 대응 출력'],
    ['J021', 'J004', '240038', 'OP-Z4', '2024-04-16 12:16:10', '2024-04-16 12:16:47', 'high', 'Z4 온도 하강 때 올라간 대응 출력'],
    ['J022', 'J004', '240038', 'TC-Z5', '2024-04-16 12:10:20', '2024-04-16 12:13:30', 'high', 'Z3 봉우리 뒤 올라온 Z5 온도'],
    ['J023', 'J004', '240038', 'OP-Z5', '2024-04-16 12:01:07', '2024-04-16 12:03:06', 'high', 'Z5 온도가 낮아질 때 올라간 출력'],
    ['J024', 'J004', '240038', 'OP-Z5', '2024-04-16 12:09:11', '2024-04-16 12:13:30', 'low', 'Z5 온도 상승 때 낮아진 출력'],
    ['J025', 'J004', '240038', 'CP-Z4', '2024-04-16 12:12:37', '2024-04-16 12:13:05', 'low', 'Z3 하강 때 반복되고 회복된 CP 하강'],
    ['J026', 'J005', '240044', 'TC-Z4', '2024-05-03 19:56:38', '2024-05-03 19:58:58', 'low', 'Z3 하강 뒤 함께 낮아진 Z4 온도'],
    ['J027', 'J005', '240044', 'OP-Z4', '2024-05-03 19:56:10', '2024-05-03 20:00:04', 'high', 'Z4 온도 하강 때 올라간 대응 출력'],
    ['J028', 'J005', '240044', 'OP-Z5', '2024-05-03 19:44:08', '2024-05-03 19:44:09', 'high', 'Z5 온도가 낮아질 때 올라간 출력'],
    ['J029', 'J005', '240044', 'OP-Z5', '2024-05-03 19:50:07', '2024-05-03 19:50:08', 'low', 'Z5 온도 상승 때 낮아진 출력'],
    ['J030', 'J005', '240044', 'CP-Z4', '2024-05-03 19:54:23', '2024-05-03 19:55:31', 'low', 'Z3 하강 때 반복되고 회복된 CP 하강'],
    ['J031', 'J005', '240044', 'TC-Z3', '2024-05-03 19:52:44', '2024-05-03 19:54:07', 'low', '다른 LOT와 같은 Z3 하강, 이후 Z4와 Z5 하강 및 Z6 하강을 확인'],
    ['J032', 'J005', '240044', 'TC-Z6', '2024-05-03 19:57:34', '2024-05-03 19:58:02', 'low', 'Z3 봉우리 뒤 늦게 내려간 Z6, 뒤 반등 일부 관측'],
    ['J033', 'J007', '240061', 'TC-Z4', '2024-06-15 03:17:34', '2024-06-15 03:19:15', 'high', 'Z3 봉우리와 가깝게 올라간 Z4 온도'],
    ['J034', 'J007', '240061', 'OP-Z4', '2024-06-15 03:16:29', '2024-06-15 03:20:37', 'low', 'Z4 온도 상승 때 낮아진 대응 출력'],
    ['J035', 'J007', '240061', 'TC-Z5', '2024-06-15 03:21:25', '2024-06-15 03:21:31', 'high', 'Z3 봉우리 뒤 올라온 Z5 온도'],
    ['J036', 'J007', '240061', 'OP-Z5', '2024-06-15 03:06:31', '2024-06-15 03:08:48', 'high', 'Z5 온도가 낮아질 때 올라간 출력'],
    ['J037', 'J007', '240061', 'OP-Z5', '2024-06-15 03:20:37', '2024-06-15 03:23:30', 'low', 'Z5 온도 상승 때 낮아진 출력'],
    ['J038', 'J007', '240061', 'CP-Z4', '2024-06-15 03:16:33', '2024-06-15 03:17:36', 'high', 'Z3 봉우리와 함께 나타나고 돌아온 짧은 CP 상승'],
    ['J039', 'J008', '240069', 'OP-Z4', '2024-07-08 05:04:23', '2024-07-08 05:05:12', 'high', 'Z4 온도 하강 때 올라간 대응 출력'],
    ['J040', 'J009', '240091', 'OP-Z4', '2024-09-09 07:12:20', '2024-09-09 07:12:21', 'low', 'Z4 온도 상승 때 낮아진 대응 출력'],
    ['J041', 'J009', '240091', 'OP-Z5', '2024-09-09 07:06:54', '2024-09-09 07:06:55', 'high', 'Z5 온도가 낮아질 때 올라간 출력'],
    ['J042', 'J009', '240091', 'CP-Z4', '2024-09-09 07:01:47', '2024-09-09 07:02:49', 'low', 'Z3 상승 전 여러 LOT에서 반복된 CP 하강'],
    ['J043', 'J009', '240091', 'CP-Z4', '2024-09-09 07:11:59', '2024-09-09 07:12:16', 'high', 'Z3 봉우리와 함께 나타나고 돌아온 짧은 CP 상승'],
    ['J044', 'J009', '240091', 'CP-Z4', '2024-09-09 07:14:26', '2024-09-09 07:17:13', 'low', 'Z3 하강 때 반복되고 회복된 CP 하강'],
    ['J045', 'J010', '240108', 'TC-Z5', '2024-10-18 10:21:46', '2024-10-18 10:21:47', 'high', 'Z3 봉우리 뒤 올라온 Z5 온도'],
    ['J046', 'J010', '240108', 'OP-Z5', '2024-10-18 10:20:43', '2024-10-18 10:21:47', 'low', 'Z5 온도 상승 때 낮아진 출력'],
    ['J047', 'J010', '240108', 'CP-Z4', '2024-10-18 10:17:09', '2024-10-18 10:17:31', 'high', 'Z3 봉우리와 함께 나타나고 돌아온 짧은 CP 상승'],
    ['J048', 'J011', '240114', 'OP-Z4', '2024-11-04 19:17:51', '2024-11-04 19:17:52', 'low', 'Z4 온도 상승 때 낮아진 대응 출력'],
    ['J049', 'J011', '240114', 'TC-Z5', '2024-11-04 19:21:21', '2024-11-04 19:25:40', 'high', 'Z3 봉우리 뒤 올라온 Z5 온도'],
    ['J050', 'J011', '240114', 'OP-Z5', '2024-11-04 19:09:50', '2024-11-04 19:09:51', 'high', 'Z5 온도가 낮아질 때 올라간 출력'],
    ['J051', 'J011', '240114', 'OP-Z5', '2024-11-04 19:20:53', '2024-11-04 19:25:06', 'low', 'Z5 온도 상승 때 낮아진 출력'],
    ['J052', 'J011', '240114', 'CP-Z4', '2024-11-04 19:09:07', '2024-11-04 19:09:31', 'low', 'Z3 상승 전 여러 LOT에서 반복된 CP 하강'],
    ['J053', 'J011', '240114', 'CP-Z4', '2024-11-04 19:19:03', '2024-11-04 19:19:40', 'high', 'Z3 봉우리와 함께 나타나고 돌아온 짧은 CP 상승'],
    ['J054', 'J011', '240114', 'CP-Z4', '2024-11-04 19:22:17', '2024-11-04 19:24:26', 'low', 'Z3 하강 때 반복되고 회복된 CP 하강'],
    ['J055', 'J012', '240117', 'TC-Z5', '2024-11-13 13:04:59', '2024-11-13 13:05:59', 'high', 'Z3 봉우리 뒤 올라온 Z5 온도'],
    ['J056', 'J012', '240117', 'OP-Z5', '2024-11-13 13:04:32', '2024-11-13 13:05:59', 'low', 'Z5 온도 상승 때 낮아진 출력'],
    ['J057', 'J013', '240121', 'TC-Z5', '2024-11-24 16:47:34', '2024-11-24 16:49:44', 'high', 'Z3 봉우리 뒤 올라온 Z5 온도'],
    ['J058', 'J013', '240121', 'OP-Z5', '2024-11-24 16:47:15', '2024-11-24 16:49:44', 'low', 'Z5 온도 상승 때 낮아진 출력'],
    ['J059', 'J013', '240121', 'CP-Z4', '2024-11-24 16:43:17', '2024-11-24 16:43:53', 'high', 'Z3 봉우리와 함께 나타나고 돌아온 짧은 CP 상승'],
    ['J060', 'J013', '240121', 'CP-Z4', '2024-11-24 16:46:30', '2024-11-24 16:48:47', 'low', 'Z3 하강 때 반복되고 회복된 CP 하강'],
    ['J061', 'J014', '240133', 'TC-Z4', '2024-12-28 15:12:30', '2024-12-28 15:13:53', 'high', 'Z3 봉우리와 가깝게 올라간 Z4 온도'],
    ['J062', 'J014', '240133', 'OP-Z4', '2024-12-28 15:19:08', '2024-12-28 15:19:09', 'high', 'Z4 온도 하강 때 올라간 대응 출력'],
    ['J063', 'J014', '240133', 'TC-Z5', '2024-12-28 15:14:50', '2024-12-28 15:17:12', 'high', 'Z3 봉우리 뒤 올라온 Z5 온도'],
    ['J064', 'J014', '240133', 'OP-Z5', '2024-12-28 15:14:50', '2024-12-28 15:17:12', 'low', 'Z5 온도 상승 때 낮아진 출력'],
    ['J065', 'J014', '240133', 'CP-Z4', '2024-12-28 15:00:44', '2024-12-28 15:01:11', 'low', 'Z3 상승 전 여러 LOT에서 반복된 CP 하강'],
    ['J066', 'J014', '240133', 'CP-Z4', '2024-12-28 15:10:14', '2024-12-28 15:11:06', 'high', 'Z3 봉우리와 함께 나타나고 돌아온 짧은 CP 상승'],
    ['J067', 'J014', '240133', 'CP-Z4', '2024-12-28 15:13:58', '2024-12-28 15:15:22', 'low', 'Z3 하강 때 반복되고 회복된 CP 하강'],
    ['J068', 'J015', '240069', 'TC-Z4', '2024-07-07 23:34:22', '2024-07-07 23:35:31', 'high', 'Z3 봉우리와 가깝게 올라간 Z4 온도'],
    ['J069', 'J015', '240069', 'OP-Z4', '2024-07-07 23:34:05', '2024-07-07 23:35:10', 'low', 'Z4 온도 상승 때 낮아진 대응 출력'],
    ['J070', 'J015', '240069', 'OP-Z5', '2024-07-07 23:33:01', '2024-07-07 23:39:05', 'low', 'Z5 온도 상승 때 낮아진 출력']
], columns=['REVIEW_ID', 'EVENT', 'LOT_NO', 'SENSOR', 'START', 'END', 'DIRECTION', 'REASON'])
joint_ranges["START"] = pd.to_datetime(joint_ranges["START"])
joint_ranges["END"] = pd.to_datetime(joint_ranges["END"])
review_ranges = pd.concat([review_ranges, joint_ranges], ignore_index=True)
display(joint_ranges[["LOT_NO", "SENSOR", "START", "END", "REASON"]])


### 4. 해당 센서의 후보만 해제하고 구역 라벨 다시 계산하기

- v3에 적힌 후보 센서에서 출발함. 새 후보를 추가하지 않음.
- 결측, 보류, 기존 제외 행은 그대로 둠.
- `TC-Z1` 후보를 해제해도 `OP-Z1` 후보가 남으면 Z1은 계속 `1`임.
- 해제 범위에 강종변경 또는 소둔로 정비 작업이 겹치면 해당 범위는 적용을 보류함.
  - 기록이 없다는 사실만으로 정상이라고 판단하지는 않음.
- 069의 지정된 Z3, Z6 범위 외에는 다른 구역 라벨을 유지함.

In [ ]:
review_groups = ["Z1", "Z2", "Z3", "Z4", "Z5", "Z6"]
review_sensors = []
for group in review_groups:
    review_sensors.extend(groups[group])
before_flags = pd.DataFrame(False, index=base.index, columns=review_sensors)
for group in review_groups:
    text = ";" + base[group + "__LABEL_SENSORS"] + ";"
    for sensor in groups[group]:
        before_flags[sensor] = text.str.contains(";" + sensor + ";", regex=False) & base[group + "__LABEL"].eq("1")

removed = before_flags.copy() & False
previous_removed = removed.copy()
joint_removed = removed.copy()
review_ids = pd.DataFrame("", index=base.index, columns=review_groups)
range_rows = []
removed_rows = []
for row in review_ranges.itertuples(index=False):
    group = row.SENSOR.split("-")[1]
    work = maintenance.loc[maintenance["STRT_DT"].lt(row.END) & maintenance["END_DT"].gt(row.START)]
    event = events.loc[events["EVT_DT"].ge(row.START) & events["EVT_DT"].lt(row.END)]
    in_range = base["LOT_NO"].eq(row.LOT_NO) & times.ge(row.START) & times.lt(row.END)
    center = lot_mean.loc[row.LOT_NO, row.SENSOR]
    same_direction = values[row.SENSOR].gt(center) if row.DIRECTION == "high" else values[row.SENSOR].lt(center)
    hit = in_range & not_excluded & before_flags[row.SENSOR] & same_direction
    apply_range = review_applied and work.empty and event.empty
    if not apply_range:
        hit = hit & False

    removed.loc[hit, row.SENSOR] = True
    if row.REVIEW_ID.startswith("J"):
        joint_removed.loc[hit, row.SENSOR] = True
    if not row.REVIEW_ID.startswith("J"):
        previous_removed.loc[hit, row.SENSOR] = True
    review_ids.loc[hit, group] += row.REVIEW_ID + ";"
    detail = base.loc[hit, ["ROW_ID", "LOT_NO", "MEAS_DT"]].copy()
    detail["GROUP"] = group
    detail["SENSOR"] = row.SENSOR
    detail["REVIEW_ID"] = row.REVIEW_ID
    removed_rows.append(detail)
    range_rows.append({
        "REVIEW_ID": row.REVIEW_ID,
        "REMOVED_SENSOR_ROWS": int(hit.sum()),
        "WORK_IDS": ";".join(work["WO_NO"].astype(str)),
        "GRADE_EVENT_IDS": ";".join(event["EVT_ID"].astype(str)),
        "APPLY_STATUS": "적용" if apply_range else "입력 변경 또는 조업 기록 겹침으로 보류",
    })

review_ranges = review_ranges.merge(pd.DataFrame(range_rows), on="REVIEW_ID", how="left")
removed_sensor_rows = pd.concat(removed_rows, ignore_index=True)
remaining_flags = before_flags & ~removed
label_change_parts = []
summary_rows = []
for group in review_groups:
    label_column = group + "__LABEL"
    old_label = base[label_column]
    touched = removed[groups[group]].any(axis=1)
    remains = remaining_flags[groups[group]].any(axis=1)
    changed = touched & old_label.eq("1") & ~remains
    previous_label = old_label.copy()
    previous_remains = (before_flags[groups[group]] & ~previous_removed[groups[group]]).any(axis=1)
    previous_label.loc[old_label.eq("1") & ~previous_remains] = "0"
    output[group + "__LABEL_BEFORE_JOINT_REVIEW"] = previous_label
    newly_cleared = previous_label.eq("1") & ~remains

    output[group + "__LABEL_V3"] = old_label
    output.loc[changed, label_column] = "0"
    output.loc[changed, group + "__LABEL_STATUS"] = "후보 아님"
    output.loc[changed, group + "__LABEL_REASON"] = "확인한 공통 움직임을 빼서 남은 후보 없음"
    output.loc[touched & remains, group + "__LABEL_REASON"] = "공통 움직임을 뺐지만 같은 구역의 다른 후보가 남음"

    remaining_sensors = pd.Series("", index=base.index)
    for sensor in groups[group]:
        remaining_sensors.loc[remaining_flags[sensor]] += sensor + ";"
    output.loc[touched, group + "__LABEL_SENSORS"] = remaining_sensors.loc[touched].str.rstrip(";")
    output.loc[touched, group + "__COMMON_REVIEW_IDS"] = base.loc[touched, group + "__COMMON_REVIEW_IDS"] + review_ids.loc[touched, group]

    # 구역별 1→0뿐 아니라 센서 후보 일부만 해제되어 1이 남은 행도 기록함.
    detail = base.loc[touched, ["ROW_ID", "LOT_NO", "MEAS_DT"]].copy()
    detail["GROUP"] = group
    detail["V3_LABEL"] = old_label.loc[touched]
    detail["V5_LABEL"] = output.loc[touched, label_column]
    detail["V3_SENSORS"] = base.loc[touched, group + "__LABEL_SENSORS"]
    detail["V5_SENSORS"] = output.loc[touched, group + "__LABEL_SENSORS"]
    detail["REVIEW_IDS"] = review_ids.loc[touched, group].str.rstrip(";")
    label_change_parts.append(detail)

    for lot in lots:
        in_lot = base["LOT_NO"].eq(lot)
        summary_rows.append({
            "LOT": lot, "구역": group,
            "직전 v5 후보 1": int((in_lot & previous_label.eq("1")).sum()),
            "이번에 1에서 0": int((in_lot & newly_cleared).sum()),
            "v3 후보 1": int((in_lot & old_label.eq("1")).sum()),
            "v5 후보 1": int((in_lot & output[label_column].eq("1")).sum()),
            "1에서 0": int((in_lot & changed).sum()),
            "일부 센서 해제, 구역 1 유지": int((in_lot & touched & remains).sum()),
            "보류 또는 제외": int((in_lot & old_label.eq("")).sum()),
        })

label_changes = pd.concat(label_change_parts, ignore_index=True).sort_values(["LOT_NO", "MEAS_DT", "GROUP"])
summary = pd.DataFrame(summary_rows).sort_values(["LOT", "구역"])
output["DATASET_VERSION"] = "v5"
output["GROUP_LABEL_VERSION"] = "v3_plus_joint_sensor_review_v5"
display(summary)
print("센서 후보 해제:", len(removed_sensor_rows), "| 구역 라벨 1→0:", int(summary["1에서 0"].sum()))
print("구역별 행수의 합임. 같은 원본 행에서 Z1과 Z2가 바뀌면 두 번 셈.")

### 4-1. 통계적 라벨과 도메인 검토 라벨 나누기

| 컬럼 예시 | 0 | 1 | 빈칸 |
| --- | --- | --- | --- |
| `Z4__STAT_LABEL` | 같은 LOT의 평균±3σ 안 | 구역 센서 중 3σ 초과가 있음 | 제외 또는 계산 불가 |
| `Z4__DOMAIN_LABEL` | 이 구역의 통계 후보를 공통 움직임으로 검토함 | 별도 근거로 이상 후보를 확인함 | 아직 판단하지 않음 |
| `Z4__LABEL` | 공통 움직임 해제 후 남은 후보 없음 | 아직 남은 통계 후보가 있음 | 제외 또는 보류 |

- `STAT_LABEL`은 검토 후에도 바꾸지 않음.
  - 평균과 표준편차는 v3처럼 해당 LOT의 제외되지 않은 관측으로 계산함.
  - LOT 전체를 살펴본 사후 통계이며 미래 예측 기준은 아님.
- `DOMAIN_LABEL=0`은 해당 통계 후보를 해제했다는 판단임.
  - 통계 후보가 없던 모든 행에 도메인 `0`을 채우지 않음.
  - 정비 기록이 없다는 이유로 `0`을 붙이지 않음.
  - 이번에는 별도 근거로 확정한 도메인 `1`이 없음.
- 모델은 기존 비교를 이어가기 위해 `__LABEL`을 사용함.
  - 따라서 결과는 검토를 반영한 통계 후보와의 일치도임.
  - 세 라벨과 판단 이유는 모델 입력에 넣지 않음.


In [ ]:
# 공통 움직임을 해제하기 전의 통계 후보를 다시 계산함.
scores = pd.DataFrame(np.nan, index=base.index, columns=sensors)
for sensor in sensors:
    mean = base["LOT_NO"].map(lot_mean[sensor])
    std = base["LOT_NO"].map(lot_std[sensor]).replace(0, np.nan)
    scores[sensor] = ((values[sensor] - mean) / std).abs().where(not_excluded)

separated_raw_rows = []
for group, group_sensors in groups.items():
    if group not in review_groups:
        output[group + "__LABEL_BEFORE_JOINT_REVIEW"] = base[group + "__LABEL"]
    available = np.isfinite(scores[group_sensors]).all(axis=1) & not_excluded
    statistical_one = scores[group_sensors].gt(3).any(axis=1)
    output[group + "__STAT_LABEL"] = np.nan
    output.loc[available, group + "__STAT_LABEL"] = statistical_one.loc[available].astype(int)

    # 남은 통계 후보가 없고 검토 번호도 있는 행만 도메인 0으로 표시함.
    reviewed = output[group + "__COMMON_REVIEW_IDS"].ne("")
    cleared = available & statistical_one & output[group + "__LABEL"].eq("0") & reviewed
    output[group + "__DOMAIN_LABEL"] = np.nan
    output.loc[cleared, group + "__DOMAIN_LABEL"] = 0
    separated_raw_rows.append({
        "구역": group,
        "통계 1": int((available & statistical_one).sum()),
        "공통 움직임으로 검토한 0": int(cleared.sum()),
        "도메인 1": 0,
        "도메인 판단 없음": int(output[group + "__DOMAIN_LABEL"].isna().sum()),
    })
separated_raw_counts = pd.DataFrame(separated_raw_rows)
audit_columns = ["ROW_ID", "LOT_NO", "MEAS_DT", "EXCLUDED"]
for group in groups:
    for suffix in ["STAT_LABEL", "DOMAIN_LABEL", "LABEL", "LABEL_BEFORE_JOINT_REVIEW", "COMMON_REVIEW_IDS"]:
        audit_columns.append(group + "__" + suffix)
raw_separate_labels = output[audit_columns].copy()
display(separated_raw_counts)


### 4-2. 069의 앞부분과 뒷부분을 함께 보기

- 파란 선은 `7/7 23:32`, 주황 선은 `7/8 04:53` 주변임.
  - Z3가 높았던 시각을 `0분`에 맞춤.
  - 각 센서의 15~10분 전 중앙값을 빼서 변화의 크기를 비교함.
- Z3, Z6뿐 아니라 Z4, Z5, CP와 대응 출력까지 비교함.
  - 같은 관계가 확인된 센서 시간만 위 표에 추가함.
  - 다른 시각인 `02:30`의 CP 급등은 해제하지 않음.


In [ ]:
# 같은 LOT에서 이미 후보 해제한 움직임과 앞부분의 움직임을 나란히 비교함.
early_times = [pd.Timestamp("2024-07-07 23:32:50"), pd.Timestamp("2024-07-08 04:53:37")]
early_sensors = ["TC-Z3", "OP-Z3", "TC-Z4", "OP-Z4", "CP-Z4", "TC-Z5", "OP-Z5", "TC-Z6", "OP-Z6"]
fig, axes = plt.subplots(3, 3, figsize=(16, 10))
for axis, sensor in zip(axes.flat, early_sensors):
    for peak, color, title in zip(early_times, ["#1765ad", "#df8d39"], ["7/7 23:32 앞부분", "7/8 04:53 기존 해제"]):
        selected = base["LOT_NO"].eq("240069") & times.ge(peak - pd.Timedelta(minutes=20)) & times.lt(peak + pd.Timedelta(minutes=30))
        block = pd.DataFrame({"TIME": times.loc[selected], "VALUE": values.loc[selected, sensor]})
        block["MINUTE"] = (block["TIME"] - peak).dt.total_seconds() / 60
        center = block.loc[block["MINUTE"].between(-15, -10), "VALUE"].median()
        # 이 그래프만 1분 평균으로 요약함. 라벨 해제 시간은 원자료 시각을 사용함.
        line = block.groupby(np.floor(block["MINUTE"]))["VALUE"].mean() - center
        axis.plot(line.index, line, color=color, label=title, linewidth=1.3)
    axis.axvline(0, color="gray", linewidth=0.6)
    axis.axhline(0, color="gray", linewidth=0.6)
    axis.set_title(sensor)
    axis.set_xlabel("Z3 봉우리에서 지난 시간, 분")
    axis.set_ylabel("봉우리 15~10분 전 중앙값과의 차이")
    axis.grid(alpha=0.2)
axes[0, 0].legend(fontsize=9)
fig.suptitle("069 앞부분과 뒷부분 | Z3~Z6, CP와 대응 출력의 관계를 함께 비교")
fig.tight_layout(rect=[0, 0, 1, 0.95])
if write_allowed:
    fig.savefig(os.path.join(FIG_DIR, "03_069_earlier_pattern_comparison.png"), dpi=130)
plt.show()
plt.close(fig)


### 4-3. 14개 LOT에서 다섯 센서가 움직이는 순서 비교

- 아래 표의 Z3 봉우리 시각을 그래프의 `0분`으로 맞춤.
  - 파란 선은 온도 또는 CP, 주황 선은 같은 구역의 출력임.
  - 센서마다 봉우리 15~10분 전 중앙값을 빼서 변화량을 표시함.
  - 초록 점은 이번 검토에서 해제한 온도 또는 CP 후보임.
- Z4는 Z3 봉우리 근처에서 움직이고, Z5는 조금 늦게 반응하는 경우가 많았음.
  - Z6는 약 10분 뒤 내려갔다가 반등하는 관계가 반복됨.
  - CP의 짧은 상승과 하강도 비슷한 시각에 나타나는 부분이 있음.
  - 어느 센서가 다른 센서를 변화시켰다는 원인 설명은 아님.
- `044`는 뒤쪽 관측이 짧아 실제로 확인한 하강과 반등 부분만 반영함.
- 아래 부분은 후보를 유지하고 도메인 판단은 비워 둠.
  - `024`, `117`: 비슷한 시각에 CP가 내려가지만 다른 LOT보다 하강이 큼.
  - `058`: CP가 내려간 뒤 계속 낮아짐.
  - `037`, `069`: 대표 움직임과 다른 시간의 CP 급등.
  - `108` 앞부분 등: Z4의 큰 변동이나 다른 센서의 회복을 같은 관계로 확인하지 못한 부분.
- 14개 LOT의 전체 선과 대표 15개 구간, 별도의 앞부분 Z3 봉우리 11개를 비교함.
  - 모든 작은 흔들림을 개별적으로 정상 판정한 것은 아님.


In [ ]:
joint_review = pd.DataFrame([
    ['J001', '240001', '2024-01-02 13:28:58', '기존 대표 움직임'],
    ['J002', '240024', '2024-03-08 04:53:32', '기존 대표 움직임'],
    ['J003', '240037', '2024-04-14 07:53:41', '기존 대표 움직임'],
    ['J004', '240038', '2024-04-16 12:07:23', '기존 대표 움직임'],
    ['J005', '240044', '2024-05-03 19:46:39', '비교할 큰 반복이 없음'],
    ['J006', '240058', '2024-06-09 07:28:05', '기존 대표 움직임'],
    ['J007', '240061', '2024-06-15 03:15:36', '기존 대표 움직임'],
    ['J008', '240069', '2024-07-08 04:53:37', '기존 대표 움직임'],
    ['J009', '240091', '2024-09-09 07:10:16', '기존 대표 움직임'],
    ['J010', '240108', '2024-10-18 10:15:44', '기존 대표 움직임'],
    ['J011', '240114', '2024-11-04 19:17:40', '기존 대표 움직임'],
    ['J012', '240117', '2024-11-13 12:58:28', '기존 대표 움직임'],
    ['J013', '240121', '2024-11-24 16:41:57', '기존 대표 움직임'],
    ['J014', '240133', '2024-12-28 15:09:48', '기존 대표 움직임'],
    ['J015', '240069', '2024-07-07 23:32:50', '앞부분에서 반복']
], columns=["EVENT", "LOT_NO", "PEAK", "NOTE"])
joint_review["PEAK"] = pd.to_datetime(joint_review["PEAK"])
joint_review["START"] = joint_review["PEAK"] - pd.Timedelta(minutes=15)
joint_review["END"] = joint_review["PEAK"] + pd.Timedelta(minutes=30)
for index, event in joint_review.iterrows():
    joint_review.loc[index, "END"] = min(event["END"], lot_end[event["LOT_NO"]])
joint_review.loc[joint_review["LOT_NO"].eq("240044"), "NOTE"] = "관측이 짧아 확인한 부분만 반영"
display(joint_review.rename(columns={"LOT_NO": "LOT", "PEAK": "Z3 봉우리", "START": "비교 시작", "END": "비교 끝", "NOTE": "설명"}))

joint_sensors = ["TC-Z3", "TC-Z4", "CP-Z4", "TC-Z5", "TC-Z6"]
paired_output = {"TC-Z3": "OP-Z3", "TC-Z4": "OP-Z4", "TC-Z5": "OP-Z5", "TC-Z6": "OP-Z6"}
fig, axes = plt.subplots(len(joint_review), 5, figsize=(22, 36))
phase_rows = []
for row_number, event in enumerate(joint_review.itertuples(index=False)):
    selected = base["LOT_NO"].eq(event.LOT_NO) & times.ge(event.START) & times.le(event.END)
    block = values.loc[selected].copy()
    relative_minutes = (times.loc[selected] - event.PEAK).dt.total_seconds() / 60
    for column_number, sensor in enumerate(joint_sensors):
        axis = axes[row_number, column_number]
        center = block.loc[relative_minutes.between(-15, -10), sensor].median()
        line = (block[sensor] - center).groupby(np.floor(relative_minutes)).mean()
        axis.plot(line.index, line, color="#1765ad", linewidth=1.1)
        cleared = joint_removed.loc[selected, sensor]
        axis.scatter(relative_minutes.loc[cleared], (block[sensor] - center).loc[cleared], color="#278447", s=3, zorder=4)
        axis.axvline(0, color="gray", linewidth=0.5)
        axis.axhline(0, color="gray", linewidth=0.5)
        axis.set_xlim(-15, 30)
        axis.grid(alpha=0.15)
        axis.set_title(sensor, fontsize=9)
        axis.tick_params(labelsize=7)
        if column_number == 0:
            axis.set_ylabel(event.LOT_NO + " / " + event.PEAK.strftime("%m/%d %H:%M:%S"), fontsize=8)
        if row_number == len(joint_review) - 1:
            axis.set_xlabel("Z3 봉우리에서 지난 분", fontsize=8)
        if sensor in paired_output:
            op = paired_output[sensor]
            op_center = block.loc[relative_minutes.between(-15, -10), op].median()
            op_line = (block[op] - op_center).groupby(np.floor(relative_minutes)).mean()
            right = axis.twinx()
            right.plot(op_line.index, op_line, color="#d98232", linewidth=0.8, alpha=0.7)
            right.tick_params(labelsize=6, colors="#ad631e")
        # 그래프와 같은 분 평균에서 상승, 하강의 크기와 시각을 표로 남김.
        for phase, start, end in [("봉우리 근처", -4, 4), ("그 뒤 4~14분", 4, 14), ("그 뒤 14~26분", 14, 26)]:
            phase_values = line.loc[(line.index >= start) & (line.index < end)].dropna()
            if len(phase_values):
                phase_rows.append({
                    "LOT_NO": event.LOT_NO, "EVENT": event.EVENT, "SENSOR": sensor, "PHASE": phase,
                    "MIN_CHANGE": phase_values.min(), "MAX_CHANGE": phase_values.max(),
                    "MIN_AT_MINUTE": phase_values.idxmin(), "MAX_AT_MINUTE": phase_values.idxmax(),
                })
fig.suptitle("14개 LOT의 다섯 센서 관계 | 파랑: TC/CP, 주황: 대응 OP, 초록: 이번에 해제한 TC/CP 후보\n왼쪽 시각이 0분. 각 센서의 15~10분 전 중앙값 대비 변화. 온도는 ℃, 출력은 %p, CP 단위는 미확인.", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.975])
if write_allowed:
    fig.savefig(os.path.join(FIG_DIR, "04_joint_sensor_review.png"), dpi=135)
plt.show()
plt.close(fig)
phase_table = pd.DataFrame(phase_rows)
display(phase_table)


### 5. 바뀐 라벨 표 바로 아래 같은 시간의 그래프 보기

- 표와 그래프 모두 실제 날짜와 시각을 사용함.
- 노란 배경: 확인한 해당 센서의 시간 범위.
- 초록 점: v3에서 후보였지만 이번에 해제한 센서값.
- 주황 점: v5에도 남은 센서 후보.
- 선은 원자료임. 센서값 자체는 바꾸지 않았음.
- `058`, `114`도 함께 보여주되 이번에는 라벨을 바꾸지 않음.

In [ ]:
range_names = {
    "SENSOR": "센서", "START": "시작, 포함", "END": "끝, 미포함",
    "DIRECTION": "방향", "REMOVED_SENSOR_ROWS": "해제한 센서 후보 수",
    "APPLY_STATUS": "적용 여부",
}
plot_sensors = ["TC-Z1", "OP-Z1", "TC-Z2", "OP-Z2"]
for lot in lots:
    print("LOT", lot, "| 마지막 기록", lot_end[lot])
    display(summary.loc[summary["LOT"].eq(lot)])
    ranges = review_ranges.loc[review_ranges["LOT_NO"].eq(lot)]
    if len(ranges):
        display(ranges[list(range_names)].rename(columns=range_names).replace({"high": "평균보다 위", "low": "평균보다 아래"}))
    else:
        print("온도와 출력 하강이 이어져 공통 움직임 해제를 적용하지 않음.")

    index = base.index[base["LOT_NO"].eq(lot) & times.ge(lot_end[lot] - pd.Timedelta(minutes=35))]
    fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)
    for axis, sensor in zip(axes.flat, plot_sensors):
        pieces = times.loc[index].diff().dt.total_seconds().gt(15).cumsum()
        for piece in pieces.unique():
            part = index[pieces.eq(piece)]
            axis.plot(times.loc[part], values.loc[part, sensor], color="#506f86", linewidth=0.7)
        selected = ranges.loc[ranges["SENSOR"].eq(sensor) & ranges["APPLY_STATUS"].eq("적용")]
        for row in selected.itertuples(index=False):
            axis.axvspan(row.START, row.END, color="#f4d368", alpha=0.25)
        dropped_index = index[removed.loc[index, sensor]]
        retained_index = index[remaining_flags.loc[index, sensor]]
        axis.scatter(times.loc[dropped_index], values.loc[dropped_index, sensor], s=4, color="#168b78", label="해제한 후보", zorder=3)
        axis.scatter(times.loc[retained_index], values.loc[retained_index, sensor], s=5, color="#dc7633", label="남은 후보", zorder=4)
        unit = "℃" if sensor.startswith("TC-") else "%"
        axis.set_title(sensor + " (" + unit + ")")
        axis.grid(alpha=0.15)
        axis.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
        axis.set_xlim(lot_end[lot] - pd.Timedelta(minutes=35), lot_end[lot])
    for axis in axes[-1]:
        axis.set_xlabel("실제 시각")
    axes[0, 0].legend(loc="best", fontsize=9)
    grade = base.loc[index[0], "GRADE"]
    fig.suptitle("LOT " + lot + " | " + grade + " | " + lot_end[lot].strftime("%Y-%m-%d") + " 마지막 35분", fontsize=15)
    fig.tight_layout(rect=[0, 0.045, 1, 0.95])
    fig.text(0.06, 0.012, "노란 배경은 확인한 센서 시간. 초록 점만 후보 해제. 다른 센서 후보와 기존 제외는 유지함.", fontsize=10)
    if write_allowed:
        fig.savefig(os.path.join(FIG_DIR, "02_LOT_" + lot + "_label_changes.png"), dpi=130)
    plt.show()
    plt.close(fig)

### 6. 원자료에서 바뀐 라벨 기록하기

- 변경 전후 라벨, 센서와 시각을 `review/v5`에 남김.
  - 이 표는 변경 근거이며 모델 입력으로 쓰지 않음.
- 원자료는 메모리에 두고 아래에서 1분으로 집계함.
  - 최종 v5는 초 단위와 분 단위를 섞지 않음.


In [ ]:
group_counts = []
for group in groups:
    label = output[group + "__LABEL"]
    status = output[group + "__LABEL_STATUS"]
    group_counts.append({
        "GROUP": group, "LABEL0": int(label.eq("0").sum()), "LABEL1": int(label.eq("1").sum()),
        "PENDING": int(status.eq("보류").sum()), "EXCLUDED": int(status.eq("제외").sum()),
    })

metadata = dict(source_metadata)
metadata["dataset_version"] = "v5"
metadata["master_file"] = OUTPUT_FILE
metadata["rows"] = len(output)
metadata["columns"] = len(output.columns)
metadata["group_counts"] = group_counts
metadata["issue"] = 108
metadata["source_sha256"] = {INPUT_FILE: source_sha256}
metadata["parent_source_sha256"] = source_metadata["source_sha256"]
for path in [SOURCE_META, EVENT_CSV, MAINT_CSV]:
    with open(path, "rb") as file:
        metadata["source_sha256"][os.path.relpath(path, ROOT)] = hashlib.file_digest(file, "sha256").hexdigest()
metadata["parent_label_policy"] = source_metadata["label_policy"]
metadata["label_policy"] = dict(source_metadata["label_policy"])
metadata["label_policy"]["version"] = "v3_plus_joint_sensor_review_v5"
metadata["label_policy"]["v5_end_review"] = {
    "reviewed_source_sha256": REVIEWED_SHA256,
    "source_matches": bool(review_applied),
    "reviewed_lots": lots,
    "sensors": review_sensors,
    "range_count": len(review_ranges),
    "time_rule": "start inclusive, end exclusive; named sensor and direction only",
    "preserve_drying_end_lots": ["240058", "240114"],
    "do_not_clear_entire_last_two_hours": True,
    "work_or_grade_event_overlap": "do not apply that range",
    "reviewed_after_observation_end": True,
    "normal_or_failure_confirmed": False,
}
metadata["review_scope"] = "Full series of 14 LOTs, 15 detailed five-channel windows, and 11 earlier Z3 peaks compared; only named sensor/time/direction ranges cleared"
metadata["model_input_forbidden"] = list(source_metadata["model_input_forbidden"]) + [group + "__LABEL_V3" for group in review_groups]
metadata["end_review_summary"] = summary.to_dict("records")
metadata["models_trained"] = False

metadata["label_separation"] = {
    "STAT_LABEL": "Same LOT nonexcluded sensor mean/std ddof0, absolute Z > 3 before common review",
    "DOMAIN_LABEL": "0 only for reviewed statistical candidates fully cleared in that group; 1 requires separate supported decision; otherwise missing",
    "LABEL": "legacy statistical candidates after common movement review; model comparison target",
    "domain_one_added_this_revision": False,
}
for group in groups:
    for suffix in ["STAT_LABEL", "DOMAIN_LABEL"]:
        metadata["model_input_forbidden"].append(group + "__" + suffix)
raw_label_output = output.copy()
raw_label_metadata = metadata.copy()
if write_allowed:
    separated_raw_counts.to_csv(os.path.join(REVIEW_DIR, "separate_raw_label_counts.csv"), index=False, encoding=ENC)
    raw_separate_labels.to_csv(os.path.join(REVIEW_DIR, "raw_separate_labels.csv.gz"), index=False, encoding=ENC)
    joint_review.to_csv(os.path.join(REVIEW_DIR, "joint_review_windows.csv"), index=False, encoding=ENC)
    phase_table.to_csv(os.path.join(REVIEW_DIR, "joint_sensor_phase_values.csv"), index=False, encoding=ENC)
    review_ranges.to_csv(os.path.join(REVIEW_DIR, "end_pattern_ranges.csv"), index=False, encoding=ENC)
    removed_sensor_rows.to_csv(os.path.join(REVIEW_DIR, "removed_sensor_rows.csv"), index=False, encoding=ENC)
    label_changes.to_csv(os.path.join(REVIEW_DIR, "label_changes.csv"), index=False, encoding=ENC)
    summary.to_csv(os.path.join(REVIEW_DIR, "label_counts_before_after.csv"), index=False, encoding=ENC)
    lot_review.to_csv(os.path.join(REVIEW_DIR, "lot_end_review.csv"), index=False, encoding=ENC)
    pd.DataFrame(group_counts).to_csv(os.path.join(REVIEW_DIR, "raw_group_label_counts.csv"), index=False, encoding=ENC)
print("원자료의 라벨 변경표를 준비함. 다음 셀에서 1분으로 묶어 최종 v5를 저장함.")


### 7. 실제 관측이 있는 분만 묶기

- `resampling`은 같은 분에 기록된 값들을 묶는 작업임.
  - 관측이 없는 날과 초는 새로 만들거나 보간하지 않음.
- LOT, 조업 상태, 제외 여부가 바뀌면 같은 분이어도 나눔.
  - 모든 센서에 `dropna()`를 적용해서 행 전체를 지우지 않음.
- `OP-Z3`가 비면 Z3 입력만 보류함.
  - 다른 구역은 그 구역에 필요한 센서가 있으면 사용함.


In [ ]:
MAX_GAP_SECONDS = 15
MIN_PAST_ROWS = 10
MIN_MINUTE_ROWS = 10
MIN_SPAN_SECONDS = 54
NOMINAL_RAW_SECONDS = 1
GRADE_GROUPS = ["Z3", "Z4", "Z5", "Z6", "Z7", "Z8", "Z9", "COOLING"]
SENSORS = source_metadata["raw_sensor_columns"]
LABEL_COLUMNS = []
for group in groups:
    for suffix in ["LABEL", "LABEL_STATUS", "LABEL_REASON", "LABEL_SENSORS", "COMMON_REVIEW_IDS", "STAT_LABEL", "DOMAIN_LABEL", "LABEL_BEFORE_JOINT_REVIEW"]:
        LABEL_COLUMNS.append(group + "__" + suffix)
files = {
    "event": EVENT_CSV,
    "maintenance": MAINT_CSV,
    "repair_plan": os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv"),
}
input_hashes = {"base": source_sha256}
for name, path in files.items():
    with open(path, "rb") as file:
        input_hashes[name] = hashlib.sha256(file.read()).hexdigest()

META = ["ROW_ID", "MEAS_DT", "LOT_NO", "GRADE", "EXCLUDED", "EXCLUSION_REASON", "DATA_KIND", "DATASET_VERSION"]
df = raw_label_output[META + SENSORS + LABEL_COLUMNS].copy()

df["MEAS_DT"] = pd.to_datetime(df["MEAS_DT"])
df["GRADE"] = df["GRADE"].fillna("미기록")
df["EXCLUDED"] = ~df["EXCLUDED"].astype("string").str.strip().str.lower().isin(["false", "0", "0.0"])
df["EXCLUSION_REASON"] = df["EXCLUSION_REASON"].fillna("")
for sensor in SENSORS:
    df[sensor] = pd.to_numeric(df[sensor], errors="coerce").replace([np.inf, -np.inf], np.nan)
for group in groups:
    for suffix in ["LABEL", "STAT_LABEL", "DOMAIN_LABEL", "LABEL_BEFORE_JOINT_REVIEW"]:
        df[group + "__" + suffix] = pd.to_numeric(df[group + "__" + suffix], errors="coerce")
    for suffix in ["LABEL_STATUS", "LABEL_REASON", "LABEL_SENSORS", "COMMON_REVIEW_IDS"]:
        column = group + "__" + suffix
        df[column] = df[column].fillna("").astype(str)

events = pd.read_csv(files["event"], encoding=ENC)
events = events.loc[events["PLANT_CD"].eq("CR1") & events["EVT_TYPE"].eq("강종변경")].copy()
events["EVT_DT"] = pd.to_datetime(events["EVT_DT"])
df = pd.merge_asof(
    df.sort_values("MEAS_DT"),
    events.sort_values("EVT_DT")[["EVT_ID", "EVT_DT", "BEF_VAL", "AFT_VAL"]],
    left_on="MEAS_DT",
    right_on="EVT_DT",
    direction="backward",
)
df = df.sort_values(["LOT_NO", "MEAS_DT"], kind="stable").reset_index(drop=True)
df["MINUTES_SINCE_GRADE_CHANGE"] = (df["MEAS_DT"] - df["EVT_DT"]).dt.total_seconds() / 60
df["EVENT_MATCH"] = df["GRADE"].eq(df["AFT_VAL"]).fillna(False) & df["EVT_DT"].notna()

maintenance = pd.read_csv(files["maintenance"], encoding=ENC)
maintenance = maintenance.loc[maintenance["EQP_TAG"].eq("P1-CR1-CAL01")].copy()
for column in ["STRT_DT", "END_DT"]:
    maintenance[column] = pd.to_datetime(maintenance[column])
df["MAINT_WORK_IDS"] = ""
for work in maintenance.itertuples(index=False):
    overlap = df["MEAS_DT"].ge(work.STRT_DT) & df["MEAS_DT"].lt(work.END_DT)
    df.loc[overlap, "MAINT_WORK_IDS"] += work.WO_NO + "|"
df["MAINT_WORK_IDS"] = df["MAINT_WORK_IDS"].str.rstrip("|")
df["MAINT_OVERLAP"] = df["MAINT_WORK_IDS"].ne("")

repair = pd.read_csv(files["repair_plan"], encoding=ENC)
repair = repair.loc[repair["PLANT_CD"].eq("CR1")].copy()
repair["STRT_DT"] = pd.to_datetime(repair["STRT_DT"])
repair["END_EXCLUSIVE"] = pd.to_datetime(repair["END_DT"]).dt.normalize() + pd.Timedelta(days=1)
df["REPAIR_PLAN_IDS"] = ""
for plan in repair.itertuples(index=False):
    overlap = df["MEAS_DT"].ge(plan.STRT_DT) & df["MEAS_DT"].lt(plan.END_EXCLUSIVE)
    df.loc[overlap, "REPAIR_PLAN_IDS"] += plan.PLAN_ID + "|"
df["REPAIR_PLAN_IDS"] = df["REPAIR_PLAN_IDS"].str.rstrip("|")

df["CONTEXT"] = "작업 기록 없음"
df.loc[df["MAINT_OVERLAP"], "CONTEXT"] = "정비 작업 기록과 겹침"
df.loc[df["REPAIR_PLAN_IDS"].ne(""), "CONTEXT"] = "정기수리 계획 기간"
df["OPERATING_NOTE"] = "별도 운전 상태 기록 없음"
df.loc[df["EXCLUDED"], "OPERATING_NOTE"] = df.loc[df["EXCLUDED"], "EXCLUSION_REASON"]
df.loc[df["EXCLUDED"] & df["LOT_NO"].eq("240058"), "OPERATING_NOTE"] = "정기수리를 위해 온도를 내림, 팀 확정"
df.loc[df["EXCLUDED"] & df["LOT_NO"].eq("240061"), "OPERATING_NOTE"] = "정기수리 후 재가동하며 온도를 올림, 팀 확정"

seconds = df["MEAS_DT"].diff().dt.total_seconds()
breaks = df["LOT_NO"].ne(df["LOT_NO"].shift()).fillna(True) | seconds.gt(MAX_GAP_SECONDS) | seconds.le(0)
for column in ["EXCLUDED", "GRADE", "EVT_ID", "CONTEXT"]:
    breaks = breaks | df[column].ne(df[column].shift()).fillna(True)
df["SEGMENT"] = breaks.cumsum()
df["BIN_START"] = df["MEAS_DT"].dt.floor("min")

# 작업번호나 제외 사유만 바뀐 경우에는 과거 60초 계산을 유지하고 분 집계만 나눔.
aggregation_keys = ["LOT_NO", "SEGMENT", "BIN_START", "EXCLUSION_REASON", "MAINT_WORK_IDS", "REPAIR_PLAN_IDS"]
df["MINUTE_ID"] = df.groupby(aggregation_keys, sort=False, dropna=False).ngroup()
print("원관측:", len(df), "/ 60초 계산 구간:", df["SEGMENT"].nunique(), "/ 1분 집계:", df["MINUTE_ID"].nunique())


### 8. 직전 60초 변화와 1분 평균 만들기

- 센서 평균, 최솟값, 최댓값, 표준편차와 기록 수를 남김.
  - 파생값은 원자료에서 계산한 뒤 1분 평균을 냄.
- `min_periods=10`은 직전 60초 기록이 10개 이상일 때 계산한다는 뜻임.
  - `closed="left"`는 현재값을 과거 평균에서 뺀다는 뜻임.
- 첫 기록부터 마지막 기록까지 54초 미만인 분은 모델 입력을 보류함.
  - 기록이 짧은 평균을 온전한 1분처럼 사용하지 않기 때문임.


In [ ]:
features = {}
for sensor in SENSORS:
    feature = pd.DataFrame(np.nan, index=df.index, columns=["VALUE", "DIFF_PER_SEC", "DEV_60S", "STD_60S"])
    for segment, block in df.loc[~df["EXCLUDED"]].groupby("SEGMENT", sort=False):
        values = pd.Series(block[sensor].to_numpy(), index=pd.DatetimeIndex(block["MEAS_DT"]))
        elapsed = values.index.to_series().diff().dt.total_seconds()
        past = values.rolling("60s", closed="left", min_periods=MIN_PAST_ROWS)
        feature.loc[block.index, "VALUE"] = values.to_numpy()
        feature.loc[block.index, "DIFF_PER_SEC"] = values.diff().div(elapsed).to_numpy()
        feature.loc[block.index, "DEV_60S"] = values.sub(past.mean()).to_numpy()
        feature.loc[block.index, "STD_60S"] = past.std().to_numpy()
    features[sensor] = feature.replace([np.inf, -np.inf], np.nan).astype("float32")

group_inputs = {}
group_valid = {}
for group, sensors in groups.items():
    parts = []
    for sensor in sensors:
        parts.append(features[sensor].add_prefix(sensor + "__"))
    group_input = pd.concat(parts, axis=1)
    if group in GRADE_GROUPS:
        group_input[group + "__GRADE_CHANGE_LOG_MIN"] = np.log1p(df["MINUTES_SINCE_GRADE_CHANGE"])
    group_input = group_input.astype("float32")
    valid = np.isfinite(group_input.to_numpy()).all(axis=1) & ~df["EXCLUDED"].to_numpy()
    if group in GRADE_GROUPS:
        valid = valid & df["EVENT_MATCH"].to_numpy(dtype=bool)
    group_inputs[group] = group_input
    group_valid[group] = valid


In [ ]:
minute_groups = df.groupby("MINUTE_ID", sort=True)
minute = minute_groups.agg(
    LOT_NO=("LOT_NO", "first"),
    SEGMENT=("SEGMENT", "first"),
    BIN_START=("BIN_START", "first"),
    OBS_START=("MEAS_DT", "min"),
    OBS_END=("MEAS_DT", "max"),
    SOURCE_FIRST_ROW_ID=("ROW_ID", "first"),
    SOURCE_LAST_ROW_ID=("ROW_ID", "last"),
    ROW_COUNT=("ROW_ID", "size"),
    GRADE=("GRADE", "first"),
    EXCLUDED=("EXCLUDED", "first"),
    EXCLUSION_REASON=("EXCLUSION_REASON", "first"),
    EVT_ID=("EVT_ID", "first"),
    EVT_DT=("EVT_DT", "first"),
    BEF_VAL=("BEF_VAL", "first"),
    AFT_VAL=("AFT_VAL", "first"),
    EVENT_MATCH=("EVENT_MATCH", "all"),
    MINUTES_SINCE_GRADE_CHANGE=("MINUTES_SINCE_GRADE_CHANGE", "mean"),
    MAINT_OVERLAP=("MAINT_OVERLAP", "first"),
    MAINT_WORK_IDS=("MAINT_WORK_IDS", "first"),
    REPAIR_PLAN_IDS=("REPAIR_PLAN_IDS", "first"),
    CONTEXT=("CONTEXT", "first"),
    OPERATING_NOTE=("OPERATING_NOTE", "first"),
)
minute.insert(0, "ROW_ID", "M" + pd.Series(np.arange(1, len(minute) + 1), index=minute.index).astype(str).str.zfill(6))
minute["BIN_END"] = minute["BIN_START"] + pd.Timedelta(minutes=1)
minute["AVAILABLE_AT"] = minute["BIN_END"]
minute["MEAS_DT"] = minute["BIN_START"]
minute["OBSERVED_SPAN_SECONDS"] = (minute["OBS_END"] - minute["OBS_START"]).dt.total_seconds() + NOMINAL_RAW_SECONDS
minute["PARTIAL_MINUTE"] = minute["OBSERVED_SPAN_SECONDS"].lt(MIN_SPAN_SECONDS)
minute["DATA_KIND"] = "minute"
minute["DATASET_VERSION"] = "v5"
minute["FEATURE_PROFILE"] = "raw60s_groupmean_1min"

sensor_tables = []
for sensor in SENSORS:
    summary = minute_groups[sensor].agg(["mean", "min", "max", "std", "count"])
    summary = summary.rename(columns={
        "mean": sensor,
        "min": sensor + "__MIN",
        "max": sensor + "__MAX",
        "std": sensor + "__STD",
        "count": sensor + "__COUNT",
    })
    sensor_tables.append(summary)
minute = pd.concat([minute] + sensor_tables, axis=1)

input_columns_by_group = {}
for group, group_input in group_inputs.items():
    valid = pd.Series(group_valid[group], index=df.index)
    available_features = group_input.copy()
    available_features.loc[~valid] = np.nan
    average_features = available_features.groupby(df["MINUTE_ID"], sort=True).mean()
    minute = pd.concat([minute, average_features], axis=1)
    input_columns_by_group[group] = group_input.columns.tolist()
    minute[group + "__FEATURE_COUNT"] = valid.groupby(df["MINUTE_ID"]).sum().astype(int)
    enough = (
        ~minute["EXCLUDED"]
        & minute["ROW_COUNT"].ge(MIN_MINUTE_ROWS)
        & ~minute["PARTIAL_MINUTE"]
        & minute[group + "__FEATURE_COUNT"].eq(minute["ROW_COUNT"])
    )
    minute[group + "__MODEL_ELIGIBLE"] = enough
    minute[group + "__MODEL_STATUS"] = "모델 입력 가능"
    minute.loc[~enough, group + "__MODEL_STATUS"] = "구역 입력이 일부 원행에서 부족함"
    minute.loc[minute["PARTIAL_MINUTE"], group + "__MODEL_STATUS"] = "관측 시간폭 " + str(MIN_SPAN_SECONDS) + "초 미만"
    minute.loc[minute["ROW_COUNT"].lt(MIN_MINUTE_ROWS), group + "__MODEL_STATUS"] = "원관측 " + str(MIN_MINUTE_ROWS) + "개 미만"
    minute.loc[minute["EXCLUDED"], group + "__MODEL_STATUS"] = "기존 제외"

group_quality = []
for group in groups:
    group_quality.append({
        "구역": group,
        "집계 분": len(minute),
        "모델 입력 가능": int(minute[group + "__MODEL_ELIGIBLE"].sum()),
        "모델 입력 보류": int((~minute[group + "__MODEL_ELIGIBLE"]).sum()),
    })
display(pd.DataFrame(group_quality).rename(columns={"집계 분": "집계 행"}))


### 9. 1분 구역 라벨 만들기

- 원자료에서 남은 후보 `1`이 있으면 해당 분의 구역 라벨은 `1`임.
  - 그 분 전체가 계속 이상했다는 뜻은 아님.
- 해당 구역의 원자료 라벨이 모두 `0`이면 `0`임.
  - 보류와 기존 제외를 `0`으로 채우지 않음.
- 후보 기록 수와 처음, 마지막 후보 시각도 남김.
  - 이 설명용 컬럼은 모델 입력에 넣지 않음.


- `STAT_LABEL`도 1분 안에 통계 후보가 하나라도 있으면 `1`임.
- `DOMAIN_LABEL=0`은 그 분의 통계 후보를 모두 공통 움직임으로 검토했다는 뜻임.
  - 나머지 초까지 전부 정상으로 확인했다는 뜻은 아님.
  - 미검토 후보나 점수 계산 불가가 남으면 비워 둠.
  - 이번에는 근거가 확정된 도메인 `1`을 새로 추가하지 않음.


In [ ]:
label_summary_rows = []
for group in groups:
    minute = minute.copy()
    label_column = group + "__LABEL"
    one = df[label_column].eq(1)
    zero = df[label_column].eq(0)
    unknown = ~(one | zero)
    minute[group + "__LABEL_1_COUNT"] = one.groupby(df["MINUTE_ID"]).sum().astype(int)
    minute[group + "__LABEL_0_COUNT"] = zero.groupby(df["MINUTE_ID"]).sum().astype(int)
    minute[group + "__LABEL_UNKNOWN_COUNT"] = unknown.groupby(df["MINUTE_ID"]).sum().astype(int)
    minute[group + "__LABEL_1_RATIO"] = minute[group + "__LABEL_1_COUNT"] / minute["ROW_COUNT"]
    minute[group + "__LABEL_FIRST_1_DT"] = df["MEAS_DT"].where(one).groupby(df["MINUTE_ID"]).min()
    minute[group + "__LABEL_LAST_1_DT"] = df["MEAS_DT"].where(one).groupby(df["MINUTE_ID"]).max()
    minute[label_column] = np.nan
    positive = ~minute["EXCLUDED"] & minute[group + "__LABEL_1_COUNT"].gt(0)
    all_zero = ~minute["EXCLUDED"] & minute[group + "__LABEL_0_COUNT"].eq(minute["ROW_COUNT"])
    minute.loc[positive, label_column] = 1
    minute.loc[all_zero, label_column] = 0
    minute[group + "__LABEL_STATUS"] = "원자료 라벨에 보류가 있음"
    minute.loc[positive, group + "__LABEL_STATUS"] = "이 분에 구역 후보가 있었음"
    minute.loc[all_zero, group + "__LABEL_STATUS"] = "원자료 구역 라벨이 모두 0"
    minute.loc[minute["EXCLUDED"], group + "__LABEL_STATUS"] = "기존 제외"
    for suffix in ["LABEL_REASON", "LABEL_SENSORS", "COMMON_REVIEW_IDS"]:
        column = group + "__" + suffix
        minute[column] = ""
        selected = df
        if suffix in ["LABEL_REASON", "LABEL_SENSORS"]:
            positive_minutes = minute.index[positive]
            selected = df.loc[~df["MINUTE_ID"].isin(positive_minutes) | one]
        source_texts = selected.groupby("MINUTE_ID")[column].unique()
        for minute_id, values in source_texts.items():
            texts = []
            for value in values:
                if value:
                    texts.append(value)
            minute.loc[minute_id, column] = " | ".join(texts)
    label_summary_rows.append({
        "구역": group,
        "원관측 라벨 1": int(one.sum()),
        "1분 라벨 1": int(positive.sum()),
        "1분 라벨 0": int(all_zero.sum()),
        "1분 라벨 보류": int(minute[label_column].isna().sum()),
        "라벨 1이지만 모델 입력 보류": int((positive & ~minute[group + "__MODEL_ELIGIBLE"]).sum()),
    })

label_summary = pd.DataFrame(label_summary_rows)
display(label_summary.rename(columns={"1분 라벨 보류": "라벨 없음(보류·기존 제외)"}))


# 통계 기록은 해제하지 않음. 도메인 검토 결과는 별도 칸에 남김.
separated_minute_rows = []
for group in groups:
    minute = minute.copy()
    statistical = df[group + "__STAT_LABEL"]
    domain = df[group + "__DOMAIN_LABEL"]
    for suffix, mask in [
        ("STAT_1_COUNT", statistical.eq(1)),
        ("STAT_0_COUNT", statistical.eq(0)),
        ("STAT_UNKNOWN_COUNT", statistical.isna()),
        ("DOMAIN_0_COUNT", domain.eq(0)),
        ("DOMAIN_1_COUNT", domain.eq(1)),
        ("DOMAIN_PENDING_CANDIDATE_COUNT", statistical.eq(1) & domain.isna()),
    ]:
        minute[group + "__" + suffix] = mask.groupby(df["MINUTE_ID"]).sum().astype(int)

    candidate = minute[group + "__STAT_1_COUNT"].gt(0)
    known = minute[group + "__STAT_UNKNOWN_COUNT"].eq(0)
    usable = ~minute["EXCLUDED"]
    minute[group + "__STAT_LABEL"] = np.nan
    minute.loc[usable & candidate, group + "__STAT_LABEL"] = 1
    minute.loc[usable & known & ~candidate, group + "__STAT_LABEL"] = 0

    all_candidates_reviewed = candidate & known & minute[group + "__DOMAIN_PENDING_CANDIDATE_COUNT"].eq(0)
    domain_one = minute[group + "__DOMAIN_1_COUNT"].gt(0)
    minute[group + "__DOMAIN_LABEL"] = np.nan
    minute.loc[usable & all_candidates_reviewed & ~domain_one, group + "__DOMAIN_LABEL"] = 0
    minute.loc[usable & domain_one, group + "__DOMAIN_LABEL"] = 1
    minute[group + "__DOMAIN_REASON"] = "도메인 판단하지 않음"
    minute.loc[usable & candidate, group + "__DOMAIN_REASON"] = "아직 검토할 후보 또는 계산 불가가 남음"
    minute.loc[usable & all_candidates_reviewed & ~domain_one, group + "__DOMAIN_REASON"] = "이 분의 통계 후보는 모두 공통 움직임으로 검토"
    minute.loc[usable & domain_one, group + "__DOMAIN_REASON"] = "근거를 확인한 이상 후보가 있음"
    minute.loc[~usable, group + "__DOMAIN_REASON"] = "기존 제외"
    separated_minute_rows.append({
        "구역": group,
        "통계 1": int(minute[group + "__STAT_LABEL"].eq(1).sum()),
        "검토 반영 후 후보 1": int(minute[group + "__LABEL"].eq(1).sum()),
        "도메인 0": int(minute[group + "__DOMAIN_LABEL"].eq(0).sum()),
        "도메인 1": int(minute[group + "__DOMAIN_LABEL"].eq(1).sum()),
        "도메인 판단 없음": int(minute[group + "__DOMAIN_LABEL"].isna().sum()),
    })
separated_minute_counts = pd.DataFrame(separated_minute_rows)
display(separated_minute_counts)


# 바로 전 v5와 비교한 1분 라벨 변경표임. 원래 v3와의 비교표와 구분함.
minute_change_rows = []
for group in groups:
    before = df[group + "__LABEL_BEFORE_JOINT_REVIEW"]
    before_one = before.eq(1).groupby(df["MINUTE_ID"]).sum()
    before_zero = before.eq(0).groupby(df["MINUTE_ID"]).sum()
    previous = pd.Series(np.nan, index=minute.index)
    previous.loc[~minute["EXCLUDED"] & before_one.gt(0)] = 1
    previous.loc[~minute["EXCLUDED"] & before_zero.eq(minute["ROW_COUNT"])] = 0
    now = minute[group + "__LABEL"]
    for lot in lots:
        selected = minute["LOT_NO"].eq(lot)
        minute_change_rows.append({
            "LOT": lot, "구역": group,
            "직전 v5 후보 1": int((selected & previous.eq(1)).sum()),
            "수정 v5 후보 1": int((selected & now.eq(1)).sum()),
            "이번에 1에서 0": int((selected & previous.eq(1) & now.eq(0)).sum()),
        })
minute_changes = pd.DataFrame(minute_change_rows)
display(minute_changes.loc[minute_changes["구역"].isin(["Z3", "Z4", "Z5", "Z6"])])
if write_allowed:
    minute_changes.to_csv(os.path.join(REVIEW_DIR, "joint_minute_label_changes.csv"), index=False, encoding=ENC)


### 10. 모델에 넣을 1분 v5 저장하기

- `data/processed/cal01_base_v5.csv`에 1분 데이터를 저장함.
  - 모델 노트북의 `DATA_FILE`을 이 파일명으로 바꾸면 됨.
- `MEAS_DT`는 분 시작, `OBS_START`와 `OBS_END`는 실제 첫 기록과 마지막 기록임.
  - `AVAILABLE_AT`은 해당 분 집계가 끝나서 사용할 수 있는 시각임.
- 구역별 `MODEL_ELIGIBLE`이 `True`인 행만 해당 구역 모델에 넣음.
  - `False`인 행도 CSV에는 남겨 보류 이유를 확인함.
- v3, v4는 덮어쓰지 않음.
  - 초 단위 결과와 분 단위 결과는 행 수가 달라 같은 분모로 비교하지 않음.


In [ ]:
row_map = df[["ROW_ID", "LOT_NO", "MEAS_DT", "SEGMENT", "MINUTE_ID"]].copy()
row_map = row_map.rename(columns={"ROW_ID": "SOURCE_ROW_ID", "MEAS_DT": "SOURCE_MEAS_DT"})
row_map["MINUTE_ROW_ID"] = row_map["MINUTE_ID"].map(minute["ROW_ID"])
row_map["BIN_START"] = row_map["MINUTE_ID"].map(minute["BIN_START"])
row_map["BIN_END"] = row_map["MINUTE_ID"].map(minute["BIN_END"])
row_map = row_map.drop(columns="MINUTE_ID")
output = minute.reset_index(drop=True)

dataset = {
    "dataset_version": "v5",
    "data_kind": "minute",
    "feature_profile": "raw60s_groupmean_1min",
    "master_file": OUTPUT_FILE,
    "source_file": INPUT_FILE,
    "source_sha256": input_hashes,
    "rows": len(output),
    "source_rows": len(df),
    "groups": groups,
    "grade_groups": GRADE_GROUPS,
    "input_columns_by_group": input_columns_by_group,
    "group_eligibility_suffix": "__MODEL_ELIGIBLE",
    "group_feature_count_suffix": "__FEATURE_COUNT",
    "label_suffix": "__LABEL",
    "aggregation": {
        "keys": aggregation_keys,
        "minute_seconds": 60,
        "observed_groups_only": True,
        "raw_sensor_fields": ["mean", "min", "max", "std_ddof1", "count"],
        "raw_mean": "원관측 산술평균, 시간 가중 평균 아님",
        "feature_mean": "같은 구역 입력이 모두 유효한 원행에서 기존 raw60초 특징의 산술평균",
        "feature_current_value": "원자료 sensor 평균과 sensor__VALUE는 입력 부족 원행 제외 여부 때문에 다를 수 있음",
        "grade_change_feature": "유효 구역 원행에서 log1p(강종변경 후 경과분)을 먼저 계산한 뒤 분 평균",
        "raw_window_seconds": 60,
        "raw_window_closed": "left",
        "raw_window_min_rows": MIN_PAST_ROWS,
        "raw_window_std_ddof": 1,
        "raw_feature_dtype_before_aggregation": "float32, 기존 IF98 특징과 동일",
        "raw_gap_seconds": MAX_GAP_SECONDS,
    },
    "model_eligibility": {
        "minimum_source_rows": MIN_MINUTE_ROWS,
        "minimum_observed_span_seconds": MIN_SPAN_SECONDS,
        "span_definition": "OBS_END-OBS_START+명목1초. 중간 결측을 채우거나 실제 관측초 수로 계산한 값은 아님.",
        "all_source_rows_must_have_group_features": True,
        "excluded_rows_eligible": False,
        "candidate_label_and_eligibility_are_separate": True,
        "minimum_values_are_domain_limits": False,
    },
    "label_policy": {
        "one": "제외되지 않은 분에 해당 구역 원라벨 1이 하나 이상 있음",
        "zero": "제외되지 않은 분의 해당 구역 원라벨이 모두 0",
        "missing": "기존 제외 또는 1이 없으면서 원라벨에 보류가 있음",
        "no_row_wide_or": True,
        "counts_and_ratio_preserved": True,
        "short_candidates_preserved": True,
        "first_last_times_do_not_fill_internal_gaps": True,
    },
    "time_policy": {
        "MEAS_DT": "BIN_START, 분 시작 표시용",
        "OBS_START": "실제 첫 원관측",
        "OBS_END": "실제 마지막 원관측",
        "AVAILABLE_AT": "BIN_END, 분이 끝난 뒤 사용 가능",
        "context": "집계 전에 원관측 시각으로 연결한 이벤트와 작업 맥락 보존",
        "split": "LOT 분할 유지. 내부 시간 분할은 OBS_START/OBS_END와 AVAILABLE_AT으로 경계 분 보류",
        "no_full_year_resampling": True,
    },
    "model_limits": [
        "분 모델은 제공된 특징을 직접 읽고 원관측 rolling을 재계산하지 않음",
        "최소 참조 256원행과 256분은 다르므로 분 모델 설정을 따로 기록",
        "raw min/max/std/count는 보존용이며 input_columns_by_group에 자동 포함하지 않음",
        "LOT 중앙60% 중앙값 기반 파생값은 미래 관측을 쓰므로 복사하지 않음",
        "정비 END_DT와 팀 제외는 사후 맥락이며 실시간 운전 명령이 아님",
        "구역 후보 비율의 단위가 초 행에서 분 행으로 바뀌므로 직접 같은 비율로 비교하지 않음",
    ],
    "row_map": "review/v5/raw_to_minute.csv",
    "issue": 108,
}

dataset["label_separation"] = raw_label_metadata["label_separation"]
dataset["separate_label_counts"] = separated_minute_counts.to_dict("records")

dataset["raw_label_review"] = raw_label_metadata["label_policy"]
dataset["raw_label_changes"] = summary_rows
dataset["reviewed_source_sha256"] = REVIEWED_SHA256
dataset["model_input_forbidden"] = raw_label_metadata["model_input_forbidden"]

for group in groups:
    for suffix in ["STAT_LABEL", "DOMAIN_LABEL", "STAT_1_COUNT", "STAT_0_COUNT", "STAT_UNKNOWN_COUNT", "DOMAIN_0_COUNT", "DOMAIN_1_COUNT", "DOMAIN_PENDING_CANDIDATE_COUNT", "DOMAIN_REASON"]:
        dataset["model_input_forbidden"].append(group + "__" + suffix)
dataset["model_input_forbidden"] = sorted(set(dataset["model_input_forbidden"]))

if write_allowed:
    separated_minute_counts.to_csv(os.path.join(REVIEW_DIR, "separate_minute_label_counts.csv"), index=False, encoding=ENC)
    os.makedirs(META_DIR, exist_ok=True)
    os.makedirs(REVIEW_DIR, exist_ok=True)
    output.to_csv(OUTPUT_CSV, index=False, encoding=ENC, float_format="%.17g")
    with open(OUTPUT_CSV, "rb") as output_file:
        dataset["master_sha256"] = hashlib.sha256(output_file.read()).hexdigest()

    # 연결표 자체의 내용으로 이름을 정해 이전 연결표를 덮지 않음
    latest_map_path = os.path.join(REVIEW_DIR, "raw_to_minute.csv")
    row_map.to_csv(latest_map_path, index=False, encoding=ENC)
    with open(latest_map_path, "rb") as map_file:
        dataset["row_map_sha256"] = hashlib.sha256(map_file.read()).hexdigest()
    row_map_name = dataset["row_map_sha256"] + "_raw_to_minute.csv"
    row_map_path = os.path.join(REVIEW_DIR, row_map_name)
    if not os.path.isfile(row_map_path):
        row_map.to_csv(row_map_path, index=False, encoding=ENC)
    dataset["row_map"] = "review/v5/" + row_map_name

    # 기존 요약 이름은 가장 최근에 만든 파일로 유지함
    label_summary.to_csv(os.path.join(REVIEW_DIR, "label_summary.csv"), index=False, encoding=ENC)
    pd.DataFrame(group_quality).to_csv(os.path.join(REVIEW_DIR, "model_input_counts.csv"), index=False, encoding=ENC)
    # 같은 CSV라도 생성 근거가 달라지면 이전 설정 원문을 따로 보존함
    metadata_text = json.dumps(dataset, ensure_ascii=False, indent=2)
    metadata_sha256 = hashlib.sha256(
        json.dumps(dataset, ensure_ascii=False, sort_keys=True).encode("utf-8")
    ).hexdigest()
    for metadata_name in ["history_" + metadata_sha256 + ".json", dataset["master_sha256"] + ".json", "dataset.json"]:
        metadata_path = os.path.join(META_DIR, metadata_name)
        if metadata_name.startswith("history_") and os.path.isfile(metadata_path):
            continue  # 같은 설정 원문은 이미 저장한 파일을 사용함
        with open(metadata_path, "w", encoding="utf-8", newline="\n") as metadata_file:
            metadata_file.write(metadata_text)
    print("저장:", OUTPUT_CSV)
    print("원자료 연결:", row_map_path)
else:
    print("저장하지 않고 집계 결과만 확인함.")
display(output.head())
